# 01. 시세 → 일드커브 (QuantLib으로 직접)

quanttool 없이 QuantLib만으로 미 국채 커브를 만든다. `src/quanttool/market/rates.py`의 `YieldCurveBuilder.build`가 내부에서 하는 일을 손으로 따라가는 것이다.

| 단계 | QuantTool 코드 | QuantLib·ORE 원본 |
|---|---|---|
| 헬퍼 | `YieldCurveBuilder._bill`, `_bond` | `references/QuantLib-SWIG/SWIG/ratehelpers.i` (`BondHelper`, `FixedRateBondHelper`) |
| 부트스트랩 | `YieldCurveBuilder.build` | `references/QuantLib-SWIG/SWIG/piecewiseyieldcurve.i` |
| 커브 빌더 설계 | 같은 곳 | `references/Engine/OREData/ored/marketdata/yieldcurve.cpp` |

> 데이터: `tests/fixtures/ust_par_sample.csv`는 재무부 CSV와 **형식만 같은 합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE_CSV = ROOT / "tests" / "fixtures" / "ust_par_sample.csv"  # 합성 샘플 (실제 시세 아님)

import pandas as pd
import QuantLib as ql

raw = pd.read_csv(SAMPLE_CSV)
raw

## 1. 하루치 시세 고르기
재무부 par yield는 **퍼센트**다. 계산에는 소수로 쓴다.

In [ ]:
row = raw[raw["Date"] == "10/23/2026"].iloc[0]
tenors = {
    "1 Mo": "1M",
    "2 Mo": "2M",
    "3 Mo": "3M",
    "4 Mo": "4M",
    "6 Mo": "6M",
    "1 Yr": "1Y",
    "2 Yr": "2Y",
    "3 Yr": "3Y",
    "5 Yr": "5Y",
    "7 Yr": "7Y",
    "10 Yr": "10Y",
    "20 Yr": "20Y",
    "30 Yr": "30Y",
}
par = {t: row[col] / 100 for col, t in tenors.items()}
par

## 2. 관례
같은 4%라도 결제일·쿠폰 주기·일수 계산이 다르면 다른 숫자다. QuantTool은 이것을 `configs/conventions/rates.yaml`의 `UST` 항목에 둔다.

In [ ]:
today = ql.Date(23, 10, 2026)
ql.Settings.instance().evaluationDate = today
cal = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
settle = cal.advance(today, 1, ql.Days)  # T+1
bond_dc = ql.ActualActual(ql.ActualActual.Bond)
settle

## 3. bill (1Y 이하): 수익률 → 할인채 가격
재무부 bill 수익률은 bond-equivalent yield(Act/365)다. 반년 이하는 단리, 넘으면 반기복리 한 번 + 남은 기간 단리. QuantTool에서는 `quanttool.market.rates.bill_price`.

In [ ]:
def bill_price(y, days):
    if days <= 182.5:
        return 100 / (1 + y * days / 365)
    h = (days - 182.5) / 365
    return 100 / ((1 + y / 2) * (1 + y * h))


helpers, labels = [], []
for t in ["1M", "2M", "3M", "4M", "6M", "1Y"]:
    mat = cal.adjust(settle + ql.Period(t), ql.Following)
    zcb = ql.ZeroCouponBond(1, cal, 100.0, mat, ql.Following, 100.0, settle)
    price = ql.QuoteHandle(ql.SimpleQuote(bill_price(par[t], mat - settle)))
    helpers.append(ql.BondHelper(price, zcb))
    labels.append(t)
len(helpers)

## 4. note·bond (2Y~30Y): par yield = 쿠폰, 가격 100
par yield의 정의가 "가격이 100이 되는 쿠폰"이므로, 쿠폰이 par yield이고 가격이 100인 가상 채권을 헬퍼로 쓴다. 지급일은 조정하지 않는다(수익률 관례, `docs/architecture.md` 1.5).

In [ ]:
for t in ["2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]:
    mat = settle + ql.Period(t)
    sch = ql.Schedule(
        settle,
        mat,
        ql.Period(ql.Semiannual),
        cal,
        ql.Unadjusted,
        ql.Unadjusted,
        ql.DateGeneration.Backward,
        False,
    )
    helpers.append(
        ql.FixedRateBondHelper(
            ql.QuoteHandle(ql.SimpleQuote(100.0)),
            1,
            100.0,
            sch,
            [par[t]],
            bond_dc,
            ql.Unadjusted,
            100.0,
            settle,
        )
    )
    labels.append(t)
len(helpers)

## 5. 부트스트랩
헬퍼 13개를 동시에 맞추는 할인 함수를 만든다. 이 객체가 "아무 날짜의 할인계수"에 답한다.

In [ ]:
curve = ql.PiecewiseLogCubicDiscount(0, cal, helpers, ql.Actual365Fixed())
curve.enableExtrapolation()

d = ql.Date(17, 3, 2031)  # 시세표에 없는 아무 날짜
print(
    "DF", curve.discount(d), " zero", curve.zeroRate(d, ql.Actual365Fixed(), ql.Continuous).rate()
)

pd.DataFrame(
    {
        "tenor": labels,
        "zero_cc_%": [
            curve.zeroRate(h.maturityDate(), ql.Actual365Fixed(), ql.Continuous).rate() * 100
            for h in helpers
        ],
        "repriced_minus_quote": [h.impliedQuote() - h.quote().value() for h in helpers],
    }
).set_index("tenor")

## 6. QuantTool로 같은 일 하기
위 1~5단계가 `Snapshot.build` 한 줄이다. 두 커브의 할인계수가 같은지 본다.

In [ ]:
from datetime import date

from quanttool.config import load_config
from quanttool.market import Snapshot, UstParCsvLoader

snap = Snapshot.build(
    date(2026, 10, 23), load_config(ROOT / "configs"), UstParCsvLoader(SAMPLE_CSV)
)
ours = snap.yield_curve("UST")
max(
    abs(
        ours.discount(today + ql.Period(m, ql.Months))
        - curve.discount(today + ql.Period(m, ql.Months))
    )
    for m in range(1, 361)
)

### 손으로 만든 커브와의 차이 하나
위 4단계는 쿠폰에 그날의 par yield를 박아 넣었다. 그러면 시세가 바뀔 때 헬퍼를 새로 만들어야 한다. QuantTool은 쿠폰을 고정하고 **가격 = f(수익률 Quote)**를 `ql.DerivedQuote`로 연결해서(`YieldCurveBuilder._bond`), Quote 값만 바꿔도 커브가 따라온다.

In [ ]:
q10 = snap.quotes["UST/PAR/10Y"]
before = ours.zeroRate(10.0, ql.Continuous).rate()
q10.setValue(q10.value() + 0.0010)
print(
    "10Y 시세 +10bp → 10Y 제로",
    round((ours.zeroRate(10.0, ql.Continuous).rate() - before) * 1e4, 2),
    "bp",
)
q10.setValue(q10.value() - 0.0010)

다음: `02_snapshot_step_by_step.ipynb` — `Snapshot.build` 안을 한 줄씩 연다.